# Prueba de funcionamiento de la búsqueda del Oráculo

Este notebook sirve para que la profesora pueda probar la búsqueda del proyecto desde Colab usando el repositorio del curso.

La idea es verificar que:
- el proyecto se descarga bien desde el repositorio,
- el modelo y los datos cargan correctamente,
- la búsqueda encuentra una configuración candidata,
- se puede validar la mejor configuración.

> Importante: este notebook no busca competir con la mejor solución; solo confirma que la búsqueda funciona en el caso base.

In [ ]:
# Celda 1: clonar el repositorio y preparar entorno
# Cambia REPO_URL por la URL de tu fork aREPO_URL ntes de ejecutar la celda.

REPO_URL = "https://github.com/NicolasJ7O/ADA-2026-2-GP01-10217"

!git clone --depth 1 {REPO_URL}
%cd Proyecto-Oraculo-AyD
!ls -la

In [ ]:
# Celda 2: instalar dependencias del proyecto
# Esto es necesario para que `ayudas.py` y `oraculo.py` puedan ejecutarse.

!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q

import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("Dependencias instaladas")

In [ ]:
# Celda 3: cargar modelo, datos y oráculo
# Si Colab no detecta GPU, debe elegir T4 antes de ejecutar esta celda.

from ayudas import cargar_modelo, cargar_datos, dividir
from oraculo import Oraculo

modelo = cargar_modelo("qwen17b")
datos = cargar_datos("datos_visibles.json")
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, lote=16, presupuesto=32_000_000)

print("Modelo y datos cargados correctamente")
print(f"Búsqueda: {len(busqueda)} instancias")
print(f"Validación: {len(validacion)} instancias")

In [ ]:
from oraculo import LIMPIA, CATALOGO
from ayudas import curva

# Estrategia eficiente para este problema: optimización por coordenadas.
# En lugar de expandir todo el árbol de combinaciones, se mejora una ranura a la
# vez sobre una muestra pequeña y luego se reevalúa la mejor config completa sobre
# toda la búsqueda. Esto reduce drásticamente el costo sin perder la estructura.
INSTANCIAS = busqueda[:20]
historial = []
RANURAS = ["rol", "estrategia", "formato", "estilo", "cierre"]

config = {r: LIMPIA[r] for r in RANURAS}
config["temperatura"] = 0.0
mejor = (0.0, dict(config))

for paso in range(4):
    mejorado = False
    for ranura in RANURAS:
        mejor_local = (0.0, dict(config))
        mejor_valor = config[ranura]

        for valor in range(len(CATALOGO[ranura])):
            if valor == config[ranura]:
                continue
            cand = dict(config)
            cand[ranura] = valor
            cand["temperatura"] = 0.0

            r = oraculo.evaluar(cand, INSTANCIAS, semilla=1)
            historial.append(r.precision)

            if r.precision > mejor_local[0]:
                mejor_local = (r.precision, cand)
                mejor_valor = valor

        if mejor_local[1] != config:
            config = dict(mejor_local[1])
            mejorado = True
            if mejor_local[0] > mejor[0]:
                mejor = (mejor_local[0], dict(config))
            print(f"paso {paso + 1}  ranura={ranura}  mejor_local={mejor_local[0]:.1%}")

    if not mejorado:
        break

# Re-evaluación final sobre toda la búsqueda con la config ya refinada.
final = oraculo.evaluar(config, busqueda, semilla=1)
mejor = (final.precision, dict(config))

print("\nmejor configuración:", mejor[1])
print(f"precisión final en búsqueda: {mejor[0]:.1%}")
curva(historial)


In [ ]:
# Celda 5: validación final de la mejor configuración
# Esta es la comprobación que la profesora puede usar para confirmar que la config funciona.

r_val = oraculo.validar(mejor[1], n=30)
print(f"Precisión validación: {r_val.precision:.1%}")
print(f"Fallos: {len(r_val.trazas)}/{r_val.n}")

## Cómo probarlo desde el local

### Opción 1: desde GitHub en Colab
1. Abre el repositorio del proyecto en GitHub.
2. Abre el notebook `prueba_busqueda_oraculo.ipynb`.
3. Haz clic en “Open in Colab”.
4. Ejecuta las celdas en orden.
5. Si la sesión lo pide, selecciona GPU T4.
6. Espera a que cargue el modelo y luego revisa la salida final.

### Opción 2: subir el notebook al entorno de Colab
1. Descarga el archivo `prueba_busqueda_oraculo.ipynb` del repositorio.
2. Abre Colab y carga el notebook.
3. Ejecuta las celdas en orden.

### Requisito clave
Los archivos del proyecto deben estar presentes en el mismo entorno de trabajo:
- `ayudas.py`
- `oraculo.py`
- `datos_visibles.json`

Sin esos archivos, la búsqueda no puede evaluarse de forma real.

> Este notebook es una prueba funcional del flujo principal del proyecto; sirve para verificar que la búsqueda está funcionando con los datos del repositorio.